Start the evalhub service:
```bash
MLFLOW_TRACKING_URI=http://localhost:5000 \
SERVICE_URL=http://localhost:8080 \
  eval-hub-server --local --configdir ./config
```

In [1]:
from evalhub import (
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
    JobStatus,
    ExperimentConfig,
    EvaluationExports,
    EvaluationExportsOCI,
    OCICoordinates,
)
import os

In [2]:
evalhub_url = "http://localhost:8080"
mlflow_url = "http://localhost:5000"

## Connect to EvalHub and check health status.

In [3]:
client = SyncEvalHubClient(base_url=evalhub_url)
health = client.health()
print(f"✓ EvalHub is healthy: {health['status']}")

✓ EvalHub is healthy: healthy


## List all providers

In [4]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

✓ Found 1 providers:
  - local_lighteval: Local LightEval


## List available benchmarks

In [5]:
provider_id = "local_lighteval"

In [6]:
benchmarks = client.benchmarks.list(provider_id=provider_id)
print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
for benchmark in benchmarks:
    print(f"  - {benchmark.id}: {benchmark.name}")


✓ Found 2 benchmarks for local_lighteval:
  - gsm8k: GSM8K
  - hellaswag: HellaSwag


## Check if the model is available

In [7]:
model_url = "http://localhost:8001/v1"
model_name = "qwen2.5-1.5b-instruct-q4_k_m.gguf"

In [8]:
import requests

response = requests.get(f"{model_url}/models")
models = response.json()
model_info = next(m for m in models["data"] if m["id"] == model_name)
print(model_info)

{'id': 'qwen2.5-1.5b-instruct-q4_k_m.gguf', 'object': 'model', 'created': 1776170452, 'owned_by': 'llamacpp', 'meta': {'vocab_type': 2, 'n_vocab': 151936, 'n_ctx_train': 32768, 'n_embd': 1536, 'n_params': 1777088000, 'size': 1111370240}}


## Submit an Evaluation Job

In [9]:
# evalhub
job_name = "ev-job1"
benchmark_id = "gsm8k"

# mlflow
experiment_name = "eval-exp1"

# oci
oci_host = "localhost:5001"
oci_repository = "eval-results"

In [13]:
from evalhub import ExperimentTag

job_request = JobSubmissionRequest(
    name=job_name,
    model=ModelConfig(url=model_url, name=model_name),
    benchmarks=[
        # BenchmarkConfig(
        #     id=benchmark_id,
        #     provider_id=provider_id,
        #     parameters={
        #         "num_examples": 10,  # Number of examples to evaluate
        #         "num_few_shot": 1,
        #     },
        # ),
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters={
                "num_examples": 10,  # Number of examples to evaluate
                "num_few_shot": 5,
            },
        )
    ],
    experiment=ExperimentConfig(
        name=experiment_name,
        description="Evaluation experiment for qwen2.5-1.5b-instruct-q4_k_m.gguf",
        tags=[
            ExperimentTag(key="model", value=model_name),
            ExperimentTag(key="provider_id", value=provider_id),
        ],
    ),
    # exports=EvaluationExports(
    #     oci=EvaluationExportsOCI(
    #         coordinates=OCICoordinates(
    #             oci_host=oci_host,
    #             oci_repository=oci_repository,
    #             annotations={
    #                 "job_name": job_name,
    #                 "experiment_name": experiment_name,
    #             },
    #         )
    #     )
    # ),
)

# Submit job using nested resource
job = client.jobs.submit(job_request)

# Store job ID for later use
job_id = job.id
print(f"✓ Job submitted successfully")
print(f"  Job ID: {job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}\n--------------\n")
print(job.model_dump_json(indent=2))

✓ Job submitted successfully
  Job ID: 4b7295c3-e815-4aba-9258-3e6969c8c193
  State: JobStatus.PENDING
  Created: 2026-04-14 18:31:44.157440+05:30
--------------

{
  "resource": {
    "id": "4b7295c3-e815-4aba-9258-3e6969c8c193",
    "tenant": null,
    "created_at": "2026-04-14T18:31:44.157440+05:30",
    "updated_at": null,
    "mlflow_experiment_id": "1",
    "message": null
  },
  "status": {
    "state": "pending",
    "message": {
      "message": "Evaluation job created",
      "message_code": "evaluation_job_created"
    },
    "benchmarks": []
  },
  "results": {
    "benchmarks": [],
    "mlflow_experiment_url": "http://localhost:5000/api/2.0/mlflow/experiments"
  },
  "name": "ev-job1",
  "description": null,
  "tags": [],
  "model": {
    "url": "http://localhost:8001/v1",
    "name": "qwen2.5-1.5b-instruct-q4_k_m.gguf",
    "auth": null
  },
  "benchmarks": [
    {
      "id": "gsm8k",
      "provider_id": "local_lighteval",
      "parameters": {
        "num_examples": 1

## Monitor the submitted job

In [14]:
import time

updated_job = client.jobs.get(job_id=job_id)

while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(job_id=job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print(f"✓ Job completed successfully: {updated_job.state}")


✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING


In [12]:
print(updated_job.model_dump_json(indent=2))

{
  "resource": {
    "id": "bb4ce0d3-4a5e-4667-924d-0ed6a06dd4c0",
    "tenant": null,
    "created_at": "2026-04-14T12:42:08Z",
    "updated_at": "2026-04-14T12:46:29Z",
    "mlflow_experiment_id": "1",
    "message": null
  },
  "status": {
    "state": "completed",
    "message": {
      "message": "Evaluation job is completed",
      "message_code": "evaluation_job_updated"
    },
    "benchmarks": [
      {
        "id": "gsm8k",
        "provider_id": "local_lighteval",
        "benchmark_index": 0,
        "status": "completed",
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-04-14T12:46:29.635333Z"
      },
      {
        "id": "gsm8k",
        "provider_id": "local_lighteval",
        "benchmark_index": 1,
        "status": "completed",
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-04-14T12:46:29.635375Z"
      }
    ]
  },
  "results": {
    "benchmarks": [
      {
        "id": "gsm8k",
   

## Check experiment runs in MLFlow

In [43]:
## MLFLOW validation
import mlflow

mlflow.set_tracking_uri(mlflow_url)

runs = mlflow.search_runs(search_all_experiments=True)

In [44]:
runs.columns

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.gsm8k_0.extractive_match',
       'metrics.all.extractive_match', 'params.duration_seconds',
       'params.benchmark_id', 'params.num_examples_evaluated',
       'params.model_name', 'tags.model', 'tags.provider_id',
       'tags.mlflow.runName'],
      dtype='object')

In [45]:
runs[["run_id", "status", "start_time", "end_time"]].head()

,run_id,status,start_time,end_time
0,8dfd4ed18b1b4eb7972c9a3613a43f9b,FINISHED,2026-04-14 11:43:56.896000+00:00,2026-04-14 11:43:56.904000+00:00
1,0f6e7a73a7a2427ead0fedfc2716bcfe,FINISHED,2026-04-14 11:43:55.650000+00:00,2026-04-14 11:43:55.658000+00:00
2,802ce2596eeb42a7bc310a6b0cc6e2fc,FINISHED,2026-04-14 09:53:05.236000+00:00,2026-04-14 09:53:05.252000+00:00
3,688af55eb4d844bc883919ac8d10dc02,FINISHED,2026-04-14 09:53:05.235000+00:00,2026-04-14 09:53:05.247000+00:00
4,1962018830024944bac5c54c1bb9e78a,FINISHED,2026-04-14 08:46:52.271000+00:00,2026-04-14 08:46:52.282000+00:00


In [46]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.gsm8k_0.extractive_match,metrics.all.extractive_match,params.duration_seconds,params.benchmark_id,params.num_examples_evaluated,params.model_name,tags.model,tags.provider_id,tags.mlflow.runName
0,8dfd4ed18b1b4eb7972c9a3613a43f9b,1,FINISHED,mlflow-artifacts:/1/8dfd4ed18b1b4eb7972c9a3613...,2026-04-14 11:43:56.896000+00:00,2026-04-14 11:43:56.904000+00:00,0.466667,0.466667,80.3554539680481,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,ac65362b-9e49-43a4-93d2-96732ae244e3
1,0f6e7a73a7a2427ead0fedfc2716bcfe,1,FINISHED,mlflow-artifacts:/1/0f6e7a73a7a2427ead0fedfc27...,2026-04-14 11:43:55.650000+00:00,2026-04-14 11:43:55.658000+00:00,0.466667,0.466667,79.10704588890076,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,ac65362b-9e49-43a4-93d2-96732ae244e3
2,802ce2596eeb42a7bc310a6b0cc6e2fc,1,FINISHED,mlflow-artifacts:/1/802ce2596eeb42a7bc310a6b0c...,2026-04-14 09:53:05.236000+00:00,2026-04-14 09:53:05.252000+00:00,0.333333,0.333333,42.710270166397095,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,cbdf2b32-da39-4f6e-bedb-536de6f112ad
3,688af55eb4d844bc883919ac8d10dc02,1,FINISHED,mlflow-artifacts:/1/688af55eb4d844bc883919ac8d...,2026-04-14 09:53:05.235000+00:00,2026-04-14 09:53:05.247000+00:00,0.400000,0.400000,42.70979189872742,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,cbdf2b32-da39-4f6e-bedb-536de6f112ad
4,1962018830024944bac5c54c1bb9e78a,1,FINISHED,mlflow-artifacts:/1/1962018830024944bac5c54c1b...,2026-04-14 08:46:52.271000+00:00,2026-04-14 08:46:52.282000+00:00,0.333333,0.333333,33.22897386550903,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,410ec1d4-ab34-4fe7-8710-7faa95dd728c
5,09148ed05dd648dc8a11f654a323e45c,1,FINISHED,mlflow-artifacts:/1/09148ed05dd648dc8a11f654a3...,2026-04-14 08:46:50.630000+00:00,2026-04-14 08:46:50.701000+00:00,0.400000,0.400000,31.58952283859253,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf,qwen2.5-1.5b-instruct-q4_k_m.gguf,local_lighteval,410ec1d4-ab34-4fe7-8710-7faa95dd728c


In [47]:
cols = [
    c
    for c in runs.columns
    if c.startswith(("metrics.", "params."))
    or c in ("run_id")
]
runs[cols].head()

,run_id,metrics.gsm8k_0.extractive_match,metrics.all.extractive_match,params.duration_seconds,params.benchmark_id,params.num_examples_evaluated,params.model_name
0,8dfd4ed18b1b4eb7972c9a3613a43f9b,0.466667,0.466667,80.3554539680481,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf
1,0f6e7a73a7a2427ead0fedfc2716bcfe,0.466667,0.466667,79.10704588890076,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf
2,802ce2596eeb42a7bc310a6b0cc6e2fc,0.333333,0.333333,42.710270166397095,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf
3,688af55eb4d844bc883919ac8d10dc02,0.400000,0.400000,42.70979189872742,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf
4,1962018830024944bac5c54c1bb9e78a,0.333333,0.333333,33.22897386550903,gsm8k,0,qwen2.5-1.5b-instruct-q4_k_m.gguf


## Check if oci artifacts are pushed

In [23]:
from oras.client import OrasClient

oras_client = OrasClient(insecure=True)

# Fetch manifest for the experiment tag
oci_image = f"{oci_host}/{oci_repository}:{oci_tag}"
manifest = oras_client.get_manifest(oci_image)
print(f"\n{oci_image} Manifest: {manifest}")



localhost:5001/eval-results:eval-exp1 Manifest: {'schemaVersion': 2, 'mediaType': 'application/vnd.oci.image.manifest.v1+json', 'artifactType': 'application/vnd.eval-hub.github.io', 'config': {'mediaType': 'application/vnd.oci.image.config.v1+json', 'size': 609, 'digest': 'sha256:c79316ae3651a9b61073b713df732419229ac9493e434e0110f7b5f3517ce1cd'}, 'layers': [{'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip', 'size': 1592, 'digest': 'sha256:609ee05778cf4326ba0cb1c1b1a52456b833a6af8484be43bb7a5f71073e6536', 'annotations': {'olot.layer.content.name': 'lighteval_results.json'}}, {'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip', 'size': 513, 'digest': 'sha256:5f8983ed4bb1838b5d335f2a4141cbc7575e60df80de5c39aa9d9966c208b53b', 'annotations': {'olot.layer.content.name': 'results.json'}}, {'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip', 'size': 389, 'digest': 'sha256:31d0b78e26a9471764ace72070ce7b7ccbf812400312d25b23d10f8f031e1f5d', 'annotations': {'olot.layer.c

In [52]:
manifest

{'schemaVersion': 2,
 'mediaType': 'application/vnd.oci.image.manifest.v1+json',
 'artifactType': 'application/vnd.eval-hub.github.io',
 'config': {'mediaType': 'application/vnd.oci.image.config.v1+json',
  'size': 609,
  'digest': 'sha256:d0e0857713bc8ac407d127a1fe0944dd49e791d80672a8b2518a18e0505e9fe8'},
 'layers': [{'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip',
   'size': 1591,
   'digest': 'sha256:8182e71359b77a512c81c19aae05106fe1d699c07a625c986cac7d20a2b5e66f',
   'annotations': {'olot.layer.content.name': 'lighteval_results.json'}},
  {'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip',
   'size': 503,
   'digest': 'sha256:2a0ad99ee91f161482163147b847573acb3abaf50800a683530cbf014eafd9b4',
   'annotations': {'olot.layer.content.name': 'results.json'}},
  {'mediaType': 'application/vnd.oci.image.layer.v1.tar+gzip',
   'size': 381,
   'digest': 'sha256:626496bd29a4eaa7bce7caccc5abf5dd1c9d893ec5742f84eef29be0e160e094',
   'annotations': {'olot.layer.content.nam

In [24]:
import tarfile
import io
import json

for layer in manifest["layers"]:
    title = layer.get("annotations", {}).get("olot.layer.content.name", "unknown")
    response = oras_client.get_blob(f"{oci_host}/{oci_repository}", layer["digest"])

    print(f"=== {title} ===")
    try:
        tar = tarfile.open(fileobj=io.BytesIO(response.content))
        for member in tar.getmembers():
            f = tar.extractfile(member)
            if f:
                content = f.read().decode("utf-8")
                if member.name.endswith(".json"):
                    print(json.dumps(json.loads(content), indent=2))
                else:
                    print(content)
        tar.close()
    except tarfile.TarError:
        print(response.text)
    print()

=== lighteval_results.json ===
{
  "config_general": {
    "lighteval_sha": "?",
    "num_fewshot_seeds": 1,
    "max_samples": 6,
    "job_id": "0",
    "start_time": 86124.330517708,
    "end_time": 86130.250771125,
    "total_evaluation_time_secondes": "5.920253416988999",
    "model_config": {
      "model_name": "openai/qwen2.5-1.5b-instruct-q4_k_m.gguf",
      "generation_parameters": {
        "num_blocks": null,
        "block_size": null,
        "early_stopping": null,
        "repetition_penalty": null,
        "frequency_penalty": null,
        "length_penalty": null,
        "presence_penalty": null,
        "max_new_tokens": null,
        "min_new_tokens": null,
        "seed": null,
        "stop_tokens": null,
        "temperature": 0,
        "top_k": null,
        "min_p": null,
        "top_p": null,
        "truncate_prompt": null,
        "cache_implementation": null,
        "response_format": null
      },
      "system_prompt": null,
      "cache_dir": "~/.cache

## Cleanup

In [54]:
client.jobs.cancel(job_id=job_id, hard_delete=True)

True

In [59]:
  # Fetch tags                                                      
tags = oras_client.get_tags(f"{oci_host}/{oci_repository}")
print("Tags:", tags)                                              


repository name not known to registry


ValueError: Issue with http://localhost:5001/v2/eval-results/tags/list: Not Found